# Assignment-7

**UCS420 “Cognitive Computing”**

**NLP-I**

**Adarsh Sharma**

**Roll No: 1024170359**

**Batch: 3Q33**

Cognitive Computing Assignment Submission

Run the cells from the provided `UCS420_LA9_Starter.ipynb` first, without changing `generate_dataset()` or `process_ticket()`. Then place these solution cells after the starter cells so the generated dataset and fingerprint remain the original ones.

## Q.1 Preprocessing

The generated tickets are processed with the supplied `process_ticket()` function. The table reports the sentence, token, punctuation, stopword, unique-token and final-token counts for every ticket. The last row totals the count-based columns.

In [ ]:
import re
import math
import nltk
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from nltk.tokenize import word_tokenize, RegexpTokenizer
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer

for resource in ["punkt", "punkt_tab", "stopwords", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng", "wordnet", "omw-1.4"]:
    try:
        nltk.download(resource, quiet=True)
    except Exception:
        pass

candidate_names = ["tickets", "raw_tickets", "ticket_data", "dataset", "texts"]
ticket_source = next((globals()[name] for name in candidate_names if name in globals()), None)

if ticket_source is None:
    raise NameError("Run the starter notebook cells first and make sure its generated ticket list is available.")

if isinstance(ticket_source, dict):
    raw_tickets = list(ticket_source.values())
elif isinstance(ticket_source, pd.DataFrame):
    text_columns = [column for column in ticket_source.columns if ticket_source[column].dtype == "object"]
    raw_tickets = ticket_source[text_columns[0]].astype(str).tolist()
else:
    raw_tickets = list(ticket_source)

raw_tickets = [str(ticket) for ticket in raw_tickets]
print("Number of tickets:", len(raw_tickets))
print("First ticket preview:", raw_tickets[0])

In [ ]:
processed_tickets = [process_ticket(ticket) for ticket in raw_tickets]
print(processed_tickets[0])

The next cell reads the common dictionary or tuple/list formats returned by `process_ticket()`. If the starter function returns a dictionary, the field names are matched by meaning; the original function is not modified.

In [ ]:
def find_field(result, words):
    if isinstance(result, dict):
        for key, value in result.items():
            normalized = str(key).lower().replace("_", " ").replace("-", " ")
            if any(word in normalized for word in words):
                return value
    return None

def as_list(value):
    if value is None:
        return []
    if isinstance(value, str):
        return value.split()
    return list(value)

def unpack_processed(result, raw_text):
    if isinstance(result, dict):
        tokens = find_field(result, ["token"])
        punctuation = find_field(result, ["punctuation", "punct"])
        stop = find_field(result, ["stopword", "stop word"])
        final = find_field(result, ["final token", "final"])
        sentences = find_field(result, ["sentence"])
        if tokens is None:
            raise ValueError("Could not identify the token list in process_ticket() output. Check the printed example above.")
        tokens = as_list(tokens)
        punctuation = as_list(punctuation) if punctuation is not None else [token for token in tokens if re.fullmatch(r"[^\w\s]+", str(token))]
        stop = as_list(stop) if stop is not None else [token for token in tokens if str(token).lower() in stop_words]
        final = as_list(final) if final is not None else [token for token in tokens if token not in punctuation and token not in stop]
        sentence_count = len(sent_tokenize(raw_text)) if sentences is None else (len(sentences) if not isinstance(sentences, (int, float)) else int(sentences))
        return sentence_count, tokens, punctuation, stop, final
    if isinstance(result, (tuple, list)) and len(result) >= 4:
        values = list(result)
        token_values = next((as_list(value) for value in values if isinstance(value, (tuple, list)) and value and all(isinstance(x, str) for x in value)), None)
        if token_values is None:
            raise ValueError("Could not identify token lists in process_ticket() output. Check the printed example above.")
        lists = [as_list(value) for value in values if isinstance(value, (tuple, list)) and all(isinstance(x, str) for x in value)]
        tokens = lists[0]
        punctuation = next((value for value in lists if value and all(re.fullmatch(r"[^\w\s]+", str(x)) for x in value)), [x for x in tokens if re.fullmatch(r"[^\w\s]+", str(x))])
        stop = next((value for value in lists if value and all(str(x).lower() in stop_words for x in value)), [x for x in tokens if str(x).lower() in stop_words])
        final = lists[-1] if len(lists) >= 4 else [x for x in tokens if x not in punctuation and x not in stop]
        return len(sent_tokenize(raw_text)), tokens, punctuation, stop, final
    raise TypeError("Unsupported process_ticket() return format. Check the printed example above.")

from nltk.tokenize import sent_tokenize
stop_words = set(stopwords.words("english"))
ticket_details = [unpack_processed(result, raw) for result, raw in zip(processed_tickets, raw_tickets)]

rows = []
for index, (sentence_count, tokens, punctuation, stop, final) in enumerate(ticket_details, start=1):
    rows.append({
        "Ticket": index,
        "Sentences": sentence_count,
        "Tokens": len(tokens),
        "Punctuation tokens": len(punctuation),
        "Stopwords": len(stop),
        "Unique tokens": len(set(tokens)),
        "Final tokens": len(final)
    })

summary = pd.DataFrame(rows)
total_row = {"Ticket": "Total"}
for column in ["Sentences", "Tokens", "Punctuation tokens", "Stopwords", "Final tokens"]:
    total_row[column] = int(summary[column].sum())
total_row["Unique tokens"] = int(summary["Unique tokens"].sum())
summary = pd.concat([summary, pd.DataFrame([total_row])], ignore_index=True)
display(summary)

selected_index = 0
selected = ticket_details[selected_index]
print("Selected ticket:", raw_tickets[selected_index])
print("Actual tokens:", selected[1])
print("Punctuation tokens:", selected[2])
print("Stopwords:", selected[3])
print("Final tokens:", selected[4])

## Q.2 Regex extraction and replacement

The expressions keep decimal numbers and version strings together, and use boundary-aware patterns for email addresses, URLs and phone numbers. The same extraction and replacement patterns are checked against `TEST_LINES` from the starter notebook.

In [ ]:
if "TEST_LINES" not in globals():
    raise NameError("Run the starter notebook cells first so TEST_LINES is defined.")

raw_corpus = "\n".join(raw_tickets)
test_text = "\n".join(str(line) for line in TEST_LINES)
regex_text = raw_corpus + "\n" + test_text

long_words_pattern = r"\b[A-Za-z]{6,}\b"
number_pattern = r"(?<![\w.])\d+(?:\.\d+)+(?![\w.])|(?<![\w.])\d+(?![\w.])"
capitalized_pattern = r"\b[A-Z][a-zA-Z]*\b"
vowel_words_pattern = r"\b[aeiouAEIOU][A-Za-z]*\b"

print("Words longer than 5 letters:", re.findall(long_words_pattern, regex_text))
print("Numbers and versions:", re.findall(number_pattern, regex_text))
print("Capitalized words:", re.findall(capitalized_pattern, regex_text))
print("Words beginning with a vowel:", re.findall(vowel_words_pattern, regex_text))

In [ ]:
email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
url_pattern = r"(?i)\b(?:https?://|www\.)[^\s<>]+"
phone_pattern = r"(?<!\w)(?:\+?\d{1,3}[ .-]?)?(?:\(?\d{3}\)?[ .-]?)\d{3}[ .-]?\d{4}(?!\w)"

def redact_text(text):
    text = re.sub(email_pattern, "<EMAIL>", text)
    text = re.sub(url_pattern, "<URL>", text)
    text = re.sub(phone_pattern, "<PHONE>", text)
    return text

print("Redacted tickets and test lines:")
for line in [*raw_tickets, *[str(item) for item in TEST_LINES]]:
    print(redact_text(line))

## Q.3 Custom tokenization

One `RegexpTokenizer` pattern is used. The alternatives are ordered so URLs, email addresses, phone numbers, version numbers, decimals, contractions and hyphenated words are captured before ordinary words and punctuation.

In [ ]:
token_pattern = r'''(?i:https?://[^\s<>]+|www\.[^\s<>]+)|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}|(?:\+?\d{1,3}[ .-]?)?(?:\(?\d{3}\)?[ .-]?)\d{3}[ .-]?\d{4}|\b\d+(?:\.\d+){2,}\b|\b\d+\.\d+\b|\b[A-Za-z]+(?:['’][A-Za-z]+)+\b|\b[A-Za-z]+(?:-[A-Za-z]+)+\b|\b[A-Za-z]+\b|\d+|[^\w\s]'''
custom_tokenizer = RegexpTokenizer(token_pattern)

def my_tokenize(text):
    return custom_tokenizer.tokenize(str(text))

comparison_lines = [str(line) for line in TEST_LINES] + raw_tickets
tokenizer_comparison = []
for index, line in enumerate(comparison_lines, start=1):
    tokenizer_comparison.append({
        "Text": line,
        "my_tokenize()": my_tokenize(line),
        "word_tokenize()": word_tokenize(line)
    })
display(pd.DataFrame(tokenizer_comparison))

### Differences between the tokenizers

- A contraction such as `isn't` is kept as one token by `my_tokenize()`, while `word_tokenize()` commonly separates it into `is` and `n't`.
- A hyphenated term such as `state-of-the-art` is kept as one token by `my_tokenize()`, while `word_tokenize()` splits it at the hyphens.
- A version such as `2.0.1` is kept as one token by `my_tokenize()`, while `word_tokenize()` may split it into numeric pieces and punctuation.
- The custom pattern also prioritizes complete email addresses, URLs and phone numbers, while the standard tokenizer may split them into several pieces.

## Q.4 Stemming and lemmatization

The comparison uses the unique final tokens produced in Q.1. POS tags are converted to WordNet parts of speech before lemmatization. The Regexp Stemmer removes the suffix `ing`, chosen because words such as `installing` and `working` commonly occur in IT-support descriptions; the cell also checks whether these example words appear in the generated dataset.

In [ ]:
final_tokens = [str(token).lower() for detail in ticket_details for token in detail[4]]
unique_final_tokens = sorted(set(token for token in final_tokens if token.isalpha()))

porter = PorterStemmer()
lancaster = LancasterStemmer()
regexp_stemmer = RegexpStemmer("ing$")
lemmatizer = WordNetLemmatizer()

def wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("N"):
        return wordnet.NOUN
    if tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

tagged_tokens = dict(nltk.pos_tag(unique_final_tokens))
stemming_rows = []
for token in unique_final_tokens:
    pos = wordnet_pos(tagged_tokens[token])
    outputs = {
        "Word": token,
        "Porter": porter.stem(token),
        "Lancaster": lancaster.stem(token),
        "Regexp": regexp_stemmer.stem(token),
        "POS-aware lemma": lemmatizer.lemmatize(token, pos)
    }
    if len(set(outputs.values())) > 1:
        stemming_rows.append(outputs)

stemming_comparison = pd.DataFrame(stemming_rows, columns=["Word", "Porter", "Lancaster", "Regexp", "POS-aware lemma"])
display(stemming_comparison)

dataset_words = set(unique_final_tokens)
print("Regexp suffix chosen: ing$")
print("Dataset examples supporting the suffix:", [word for word in ["installing", "working", "running", "loading", "updating"] if word in dataset_words])

## Q.5 Corpus statistics

S1 contains all tokens, S2 removes punctuation tokens, S3 uses the final tokens from preprocessing, and S4 applies Porter stemming to S3. Type-token ratio (TTR) is the number of unique tokens divided by the total number of tokens.

In [ ]:
all_tokens = [str(token).lower() for detail in ticket_details for token in detail[1]]
punctuation_tokens = [str(token) for detail in ticket_details for token in detail[2]]
s1 = all_tokens
s2 = [token for token in s1 if not re.fullmatch(r"[^\w\s]+", token)]
s3 = final_tokens
s4 = [porter.stem(token) for token in s3]

corpora = {"S1: All tokens": s1, "S2: Without punctuation": s2, "S3: Final tokens": s3, "S4: Porter-stemmed S3": s4}
statistics_rows = []
for name, corpus in corpora.items():
    total = len(corpus)
    unique = len(set(corpus))
    statistics_rows.append({"Corpus": name, "Total tokens": total, "Unique tokens": unique, "TTR": unique / total if total else 0})
display(pd.DataFrame(statistics_rows))

In [ ]:
print("Top 10 words in S1:")
display(pd.DataFrame(Counter(s1).most_common(10), columns=["Word", "Frequency"]))

print("Top 10 words in S3:")
display(pd.DataFrame(Counter(s3).most_common(10), columns=["Word", "Frequency"]))

frequencies = Counter(s3)
ranked_frequencies = sorted(frequencies.values(), reverse=True)
ranks = list(range(1, len(ranked_frequencies) + 1))
plt.figure(figsize=(8, 5))
plt.plot([math.log(rank) for rank in ranks], [math.log(freq) for freq in ranked_frequencies], marker="o", linestyle="-")
plt.xlabel("log(rank)")
plt.ylabel("log(frequency)")
plt.title("Log-Rank vs Log-Frequency for S3")
plt.grid(True, alpha=0.3)
plt.show()

print("Top 5 bigrams in S1:")
display(pd.DataFrame(Counter(nltk.ngrams(s1, 2)).most_common(5), columns=["Bigram", "Frequency"]))

print("Top 5 bigrams in S3:")
display(pd.DataFrame(Counter(nltk.ngrams(s3, 2)).most_common(5), columns=["Bigram", "Frequency"]))

## Q.6 Interpretation

**(a) Why does TTR change from S1 to S4?**

TTR changes because S1 includes punctuation and the original token forms, while S4 contains final tokens after preprocessing and Porter stemming. Removing punctuation and stopwords changes the total and unique-token counts. Stemming also combines related forms such as `connect`, `connected` and `connecting` into fewer stems, which can lower the number of unique tokens. Since TTR is unique tokens divided by total tokens, the direction and size of the change depend on the generated tickets.

**(b) One difference between `word_tokenize()` and the custom tokenizer**

For a contraction such as `isn't`, the custom tokenizer keeps the full contraction as one token. `word_tokenize()` commonly splits it into `is` and `n't`, because its standard rules separate the contraction suffix.

**(c) One undesirable stemming result**

A stemmer may reduce a valid word to a fragment that is not a standalone word. For example, Porter stemming can reduce `studies` to `studi`. The stem is useful for grouping related words but is not a natural word form.

**(d) POS-aware lemmatization example**

For the verb `running`, POS-aware lemmatization can return `run`, which is a valid base form. Stemming may return a shortened fragment such as `run` or another stem depending on the algorithm, without using grammatical context. POS information helps the lemmatizer choose the correct dictionary form for the token's role in the sentence.

The dataset-dependent counts, examples and tables above are generated from the tickets created by the supplied starter notebook.